In [123]:
import torch
import transformers
from transformers.trainer_pt_utils import LabelSmoother
IGNORE_TOKEN_ID = LabelSmoother.ignore_index

TEST_SOURCES = [
     [
            {
                "role": "system",
                "content": "System says hello."
            },
            {
                "role": "user",
                "content": "A says marco"
            },
            {
                "role": "assistant",
                "content": "A says polo"
            },
            {
                "role": "user",
                "content": "U says gamma"
            },
            {
                "role": "assistant",
                "content": "A says delta"
            }
        ],
    [
            # {
            #     "role": "system",
            #     "content": "System says hello2."
            # },
            {
                "role": "user",
                "content": "User says marco2"
            },
            {
                "role": "assistant",
                "content": "Assistant says polo2"
            },
            {
                "role": "user",
                "content": "User says gamma2"
            },
            {
                "role": "assistant",
                "content": "Assistant says delta2"
            }
    ]
    
]

# Args
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
training_cache_dir = "/data/tir/projects/tir6/bisk/athankar/projects/.cache"
training_model_max_length = 64

config = transformers.AutoConfig.from_pretrained(
    model_name,
    trust_remote_code=True,
    cache_dir=training_cache_dir,
)
config.use_cache = False

# Load tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(
        model_name,
        config=config,
        trust_remote_code=True,
        cache_dir=training_cache_dir,
        model_max_length=training_model_max_length,
        padding_side="right",
        use_fast=False,)

tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

llama3btokenizer = transformers.AutoTokenizer.from_pretrained(
        "meta-llama/Meta-Llama-3-8.1B-Instruct",
        config=config,
        trust_remote_code=True,
        cache_dir=training_cache_dir,
        model_max_length=training_model_max_length,
        padding_side="right",
        use_fast=False,)


In [124]:
chat_template_formatted = tokenizer.apply_chat_template(TEST_SOURCES, tokenize=False, padding="max_length")

In [125]:
chat_template_formatted

['<s> [INST] System says hello.\n\nA says marco [/INST] A says polo</s> [INST] U says gamma [/INST] A says delta</s>',
 '<s> [INST] User says marco2 [/INST] Assistant says polo2</s> [INST] User says gamma2 [/INST] Assistant says delta2</s>']

In [126]:
chat_template_formatted_tokens = tokenizer.apply_chat_template(TEST_SOURCES, tokenize=True, padding="max_length", return_tensors="pt")

In [127]:
chat_template_formatted_tokens

tensor([[    1,   733, 16289, 28793,  2135,  2627,  6312, 28709, 28723,    13,
            13, 28741,  2627,  1829,  1115,   733, 28748, 16289, 28793,   330,
          2627,  1160, 28709,     2,   733, 16289, 28793,   500,  2627,   319,
          2653,   733, 28748, 16289, 28793,   330,  2627, 14823,     2,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [    1,   733, 16289, 28793,  1247,  2627,  1829,  1115, 28750,   733,
         28748, 16289, 28793, 21631,  2627,  1160, 28709, 28750,     2,   733,
         16289, 28793,  1247,  2627,   319,  2653, 28750,   733, 28748, 16289,
         28793, 21631,  2627, 14823, 28750,     2,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0

In [128]:
tokenizer.eos_token, tokenizer.eos_token_id

('</s>', 2)

In [129]:
tokenizer.chat_template

"{%- if messages[0]['role'] == 'system' %}\n    {%- set system_message = messages[0]['content'] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set loop_messages = messages %}\n{%- endif %}\n\n{{- bos_token }}\n{%- for message in loop_messages %}\n    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}\n        {{- raise_exception('After the optional system message, conversation roles must alternate user/assistant/user/assistant/...') }}\n    {%- endif %}\n    {%- if message['role'] == 'user' %}\n        {%- if loop.first and system_message is defined %}\n            {{- ' [INST] ' + system_message + '\\n\\n' + message['content'] + ' [/INST]' }}\n        {%- else %}\n            {{- ' [INST] ' + message['content'] + ' [/INST]' }}\n        {%- endif %}\n    {%- elif message['role'] == 'assistant' %}\n        {{- ' ' + message['content'] + eos_token}}\n    {%- else %}\n        {{- raise_exception('Only user and assistant roles are supported, with the exc

In [130]:
chat_template_formatted

['<s> [INST] System says hello.\n\nA says marco [/INST] A says polo</s> [INST] U says gamma [/INST] A says delta</s>',
 '<s> [INST] User says marco2 [/INST] Assistant says polo2</s> [INST] User says gamma2 [/INST] Assistant says delta2</s>']

In [131]:
toks =  tokenizer(
   chat_template_formatted,
   return_tensors="pt",
   padding="max_length",
   max_length=tokenizer.model_max_length,
   truncation=True,
   return_special_tokens_mask=True,
   add_special_tokens=False
      )

In [132]:
tokenizer.decode(toks["input_ids"][0])

'<s> [INST] System says hello.\n\nA says marco [/INST] A says polo</s> [INST] U says gamma [/INST] A says delta</s><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk>'

In [133]:
toks

{'input_ids': tensor([[    1,   733, 16289, 28793,  2135,  2627,  6312, 28709, 28723,    13,
            13, 28741,  2627,  1829,  1115,   733, 28748, 16289, 28793,   330,
          2627,  1160, 28709,     2,   733, 16289, 28793,   500,  2627,   319,
          2653,   733, 28748, 16289, 28793,   330,  2627, 14823,     2,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [    1,   733, 16289, 28793,  1247,  2627,  1829,  1115, 28750,   733,
         28748, 16289, 28793, 21631,  2627,  1160, 28709, 28750,     2,   733,
         16289, 28793,  1247,  2627,   319,  2653, 28750,   733, 28748, 16289,
         28793, 21631,  2627, 14823, 28750,     2,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,


In [134]:
tokenizer.decode(tokenizer.apply_chat_template(TEST_SOURCES, 
                                               tokenize=True, 
                                               padding="max_length", 
                                               return_tensors="pt", 
                                              add_generation_prompt=True)[0])

'<s> [INST] System says hello.\n\nA says marco [/INST] A says polo</s> [INST] U says gamma [/INST] A says delta</s><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk>'

In [135]:
tokenizer.decode(tokenizer.apply_chat_template(TEST_SOURCES, 
                                               tokenize=True, 
                                               padding="max_length", 
                                               return_tensors="pt", 
                                              add_generation_prompt=False)[0])

'<s> [INST] System says hello.\n\nA says marco [/INST] A says polo</s> [INST] U says gamma [/INST] A says delta</s><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk>'

In [136]:

tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id
tokenizer.apply_chat_template(TEST_SOURCES, tokenize=True, padding="max_length", return_tensors="pt",)[0]

tensor([    1,   733, 16289, 28793,  2135,  2627,  6312, 28709, 28723,    13,
           13, 28741,  2627,  1829,  1115,   733, 28748, 16289, 28793,   330,
         2627,  1160, 28709,     2,   733, 16289, 28793,   500,  2627,   319,
         2653,   733, 28748, 16289, 28793,   330,  2627, 14823,     2,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])

In [137]:
tokenizer.chat_template

"{%- if messages[0]['role'] == 'system' %}\n    {%- set system_message = messages[0]['content'] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set loop_messages = messages %}\n{%- endif %}\n\n{{- bos_token }}\n{%- for message in loop_messages %}\n    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}\n        {{- raise_exception('After the optional system message, conversation roles must alternate user/assistant/user/assistant/...') }}\n    {%- endif %}\n    {%- if message['role'] == 'user' %}\n        {%- if loop.first and system_message is defined %}\n            {{- ' [INST] ' + system_message + '\\n\\n' + message['content'] + ' [/INST]' }}\n        {%- else %}\n            {{- ' [INST] ' + message['content'] + ' [/INST]' }}\n        {%- endif %}\n    {%- elif message['role'] == 'assistant' %}\n        {{- ' ' + message['content'] + eos_token}}\n    {%- else %}\n        {{- raise_exception('Only user and assistant roles are supported, with the exc

In [138]:
tokenizer.unk_token

'<unk>'

In [139]:
tokenizer.apply_chat_template(

SyntaxError: incomplete input (2924293870.py, line 1)

In [141]:
tokenizer.apply_chat_template(TEST_SOURCES, tokenize=False, padding="max_length")

['<s> [INST] System says hello.\n\nA says marco [/INST] A says polo</s> [INST] U says gamma [/INST] A says delta</s>',
 '<s> [INST] User says marco2 [/INST] Assistant says polo2</s> [INST] User says gamma2 [/INST] Assistant says delta2</s>']

In [142]:
llama3btokenizer.apply_chat_template(TEST_SOURCES, tokenize=False, padding="max_length")

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nSystem says hello.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nA says marco<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nA says polo<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nU says gamma<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nA says delta<|eot_id|>',
 '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nUser says marco2<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAssistant says polo2<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nUser says gamma2<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAssistant says delta2<|eot_id|>']

In [146]:
llama3btokenizer.mask_token_id

In [144]:
llama3btokenizer.bos_token

'<|begin_of_text|>'